# Comparison with Pieter's RNN-LSTM Results

## Re-evaluating our CQR-XGBQ forecast using Pieter's methodology

Pieter Hernando C.S. (2023) used an RNN-LSTM to forecast GHI, temperature, and load demand
at the NTUST site. His evaluation (Table 4.1) uses a **per-season split** with **MAE, MAPE, RMSE**
computed over **all hours** (including nighttime zeros).

### Key differences from our standard evaluation:

| Aspect | Our Pipeline | Pieter's Pipeline |
|--------|-------------|-------------------|
| Model | CQR-XGBQ (XGBoost quantile) | RNN-LSTM (dual-layer) |
| Output | 19 quantiles (P05–P95) | Point forecast only |
| Data period | 2021–2025 CWA+GFS | 2019 CWA only |
| Train/test split | Chronological (last 365 days = test) | Per-season (train/test within each season) |
| Evaluation scope | All hours + daytime-only | All hours only |
| Metrics | MAE, RMSE, R², CRPS, PICP | MAE, MAPE, RMSE |
| Seasons | Not used | Summer (Jun–Sep), Fall (Oct–Nov), Winter (Dec–Mar), Spring (Apr–May) |
| NWP features | Yes (14 GFS features) | No (only historical GHI, temp, load) |

### Pieter's season definitions:
- **Summer**: June – September
- **Fall**: October – November  
- **Winter**: December – March
- **Spring**: April – May

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Load our CQR-calibrated forecast (test set)
forecast = pd.read_parquet('../pipeline_outputs/forecast_ghi_quantiles_daily.parquet')
test = forecast[forecast['split_name'] == 'test'].copy()

# Extract month from target_time_local
test['month'] = pd.to_datetime(test['target_time_local']).dt.month

# Pieter's season mapping
def get_pieter_season(month):
    if month in [6, 7, 8, 9]:
        return 'Summer'
    elif month in [10, 11]:
        return 'Fall'
    elif month in [12, 1, 2, 3]:
        return 'Winter'
    elif month in [4, 5]:
        return 'Spring'

test['season'] = test['month'].apply(get_pieter_season)

print(f'Test set: {len(test)} hours')
print(f'Date range: {test["target_time_local"].min()} to {test["target_time_local"].max()}')
print(f'\nSeason distribution:')
print(test['season'].value_counts())

## Compute Per-Season Metrics (Pieter's Way)

MAE, MAPE, RMSE — all hours (including nighttime zeros), per season.
MAPE for GHI is not reported by Pieter (undefined when actual=0).

In [ ]:
def compute_metrics_pieter_style(y_true, y_pred):
    """Compute MAE, MAPE, RMSE exactly as Pieter does (all hours)."""
    mae = np.mean(np.abs(y_true - y_pred))
    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))
    # MAPE: Pieter doesn't report for GHI (too many zeros), but we compute where possible
    nonzero = y_true > 0
    if nonzero.sum() > 0:
        mape = np.mean(np.abs((y_true[nonzero] - y_pred[nonzero]) / y_true[nonzero]))
    else:
        mape = np.nan
    return mae, mape, rmse

# Also compute R² for our own reference (Pieter doesn't report this)
def compute_r2(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    return 1 - ss_res / ss_tot if ss_tot > 0 else np.nan

# --- Our results per season (all hours, using q0.50 as point forecast) ---
y_true = test['label_ghi_obs_wm2'].values
y_pred = test['q0.50'].values

seasons_order = ['Summer', 'Fall', 'Winter', 'Spring']
our_results = []

for season in seasons_order:
    mask = test['season'] == season
    yt = y_true[mask]
    yp = y_pred[mask]
    mae, mape, rmse = compute_metrics_pieter_style(yt, yp)
    r2 = compute_r2(yt, yp)
    n_hours = mask.sum()
    our_results.append({
        'Season': season, 'N_hours': n_hours,
        'MAE': mae, 'MAPE': mape, 'RMSE': rmse, 'R2': r2,
    })

df_ours = pd.DataFrame(our_results)

# --- Pieter's results from Table 4.1 (Test columns for GHI) ---
pieter_results = [
    {'Season': 'Summer', 'MAE': 105.1273, 'MAPE': np.nan, 'RMSE': 154.6486},
    {'Season': 'Fall',   'MAE': 93.56821, 'MAPE': np.nan, 'RMSE': 136.0632},
    {'Season': 'Winter', 'MAE': 99.80935, 'MAPE': np.nan, 'RMSE': 142.0352},
    {'Season': 'Spring', 'MAE': 118.7704, 'MAPE': np.nan, 'RMSE': 158.1262},
]
df_pieter = pd.DataFrame(pieter_results)

print('Our GHI forecast metrics (per season, all hours — Pieter\'s methodology):')
print(df_ours.to_string(index=False, float_format='%.2f'))
print()
print('Pieter\'s GHI forecast metrics (Table 4.1, Test):')
print(df_pieter.to_string(index=False, float_format='%.2f'))

## Side-by-Side Comparison Table

In [ ]:
# ============================================================
# SIDE-BY-SIDE COMPARISON: Our CQR-XGBQ vs Pieter's RNN-LSTM
# ============================================================
print('='*85)
print('  GHI FORECAST COMPARISON: CQR-XGBQ (Ours) vs RNN-LSTM (Pieter)')
print('  Metrics: MAE & RMSE in W/m², all hours (including nighttime zeros)')
print('='*85)
print()

header = f'{"Season":<10s}  {"MAE (Ours)":>10s} {"MAE (Pieter)":>12s} {"Delta":>8s} {"Improve":>8s}  |  {"RMSE (Ours)":>11s} {"RMSE (Pieter)":>13s} {"Delta":>8s} {"Improve":>8s}'
print(header)
print('-' * len(header))

for i, season in enumerate(seasons_order):
    our = df_ours.iloc[i]
    piet = df_pieter.iloc[i]
    
    mae_delta = our['MAE'] - piet['MAE']
    mae_pct = mae_delta / piet['MAE'] * 100
    rmse_delta = our['RMSE'] - piet['RMSE']
    rmse_pct = rmse_delta / piet['RMSE'] * 100
    
    mae_sign = '+' if mae_delta > 0 else ''
    rmse_sign = '+' if rmse_delta > 0 else ''
    
    print(f'{season:<10s}  {our["MAE"]:10.2f} {piet["MAE"]:12.2f} {mae_sign}{mae_delta:7.2f} {mae_pct:+7.1f}%  |  {our["RMSE"]:11.2f} {piet["RMSE"]:13.2f} {rmse_sign}{rmse_delta:7.2f} {rmse_pct:+7.1f}%')

# Overall (annual average)
our_mae_avg = df_ours['MAE'].mean()
our_rmse_avg = df_ours['RMSE'].mean()
piet_mae_avg = df_pieter['MAE'].mean()
piet_rmse_avg = df_pieter['RMSE'].mean()

print('-' * len(header))
mae_d = our_mae_avg - piet_mae_avg
rmse_d = our_rmse_avg - piet_rmse_avg
ms = '+' if mae_d > 0 else ''
rs = '+' if rmse_d > 0 else ''
print(f'{"Avg":<10s}  {our_mae_avg:10.2f} {piet_mae_avg:12.2f} {ms}{mae_d:7.2f} {mae_d/piet_mae_avg*100:+7.1f}%  |  {our_rmse_avg:11.2f} {piet_rmse_avg:13.2f} {rs}{rmse_d:7.2f} {rmse_d/piet_rmse_avg*100:+7.1f}%')

print()
print('Negative delta = our model is better. Positive = Pieter is better.')
print()

# Additional: our R² per season (Pieter doesn't report this)
print('='*50)
print('  Our R² per season (not available from Pieter)')
print('='*50)
for i, season in enumerate(seasons_order):
    print(f'  {season:<10s}: R² = {df_ours.iloc[i]["R2"]:.4f}')

## Daylight-Only Comparison

Pieter's all-hours MAE is diluted by nighttime zeros (both models predict ~0 at night).
This section shows the daylight-only comparison for a more meaningful signal.

In [ ]:
# Daylight-only metrics per season
daylight_mask = test['solar_elevation'].values > 0

print('='*70)
print('  Our GHI Forecast — DAYLIGHT ONLY (solar_elevation > 0)')
print('='*70)
print(f'{"Season":<10s}  {"N_day":>6s}  {"MAE":>8s}  {"RMSE":>8s}  {"R²":>8s}')
print('-' * 50)

our_daylight = []
for season in seasons_order:
    mask = (test['season'] == season).values & daylight_mask
    yt = y_true[mask]
    yp = y_pred[mask]
    mae = np.mean(np.abs(yt - yp))
    rmse = np.sqrt(np.mean((yt - yp) ** 2))
    r2 = compute_r2(yt, yp)
    print(f'{season:<10s}  {mask.sum():6d}  {mae:8.2f}  {rmse:8.2f}  {r2:8.4f}')
    our_daylight.append({'Season': season, 'MAE_day': mae, 'RMSE_day': rmse, 'R2_day': r2})

# Overall daylight
yt_all = y_true[daylight_mask]
yp_all = y_pred[daylight_mask]
mae_all = np.mean(np.abs(yt_all - yp_all))
rmse_all = np.sqrt(np.mean((yt_all - yp_all) ** 2))
r2_all = compute_r2(yt_all, yp_all)
print('-' * 50)
print(f'{"Overall":<10s}  {daylight_mask.sum():6d}  {mae_all:8.2f}  {rmse_all:8.2f}  {r2_all:8.4f}')

print()
print('Note: Pieter does not report daylight-only metrics, so this is our-only.')
print('His all-hours numbers are inflated by easy nighttime zeros.')

## Summary & Caveats

### Important caveats for this comparison:
1. **Different data years**: We use 2024–2025 test data; Pieter used 2019. Weather patterns differ year-to-year.
2. **Different training approaches**: Pieter trains a separate model per season; we train one model on all data.
3. **Pieter uses no NWP**: His LSTM uses only historical GHI/temp/load as inputs — no weather forecast data.
4. **Our model is day-ahead**: We forecast D+1 using D-1 information only. Pieter's LSTM uses recent history (t-n to t) which may include same-day observations.
5. **Different data sources**: Pieter likely used a different CWA station or dataset version.

Despite these differences, the per-season MAE/RMSE comparison using identical metric definitions provides a fair methodological alignment.